# Azure AI Agent Framework - Basic Example

This notebook demonstrates basic usage of `AzureAIProjectAgentProvider` to create agents with function tools. You'll learn how to:
- Create an agent with custom instructions
- Add function tools to extend agent capabilities
- Use both streaming and non-streaming responses

## Prerequisites
- Azure AI Project created in Azure AI Foundry
- `AZURE_AI_PROJECT_ENDPOINT` environment variable set
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` environment variable (optional, defaults to gpt-4o)
- Azure CLI credentials configured (`az login`)

## Setup: Import Required Libraries

In [1]:
# Copyright (c) Microsoft. All rights reserved.

import asyncio
import os
from random import randint
from typing import Annotated

from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity.aio import DefaultAzureCredential
from dotenv import load_dotenv
from pydantic import Field

# Enable nested asyncio for Jupyter notebooks
import nest_asyncio
nest_asyncio.apply()

# Load environment variables
load_dotenv()

print("✅ All imports loaded successfully")
print("✅ Async support enabled for Jupyter")

✅ All imports loaded successfully
✅ Async support enabled for Jupyter


## Setup: Create Agent Instructions

In [2]:
agent_instructions = """
You are a friendly and highly professional AI Agent that can help plan vacations for customers at the following destinations:
    1. "Barcelona, Spain",
    2. "Paris, France", 
    3. "Berlin, Germany",
    4. "Tokyo, Japan",
    5. "Sydney, Australia",
    6. "New York, USA",
    7. "Cairo, Egypt",
    8. "Cape Town, South Africa",
    9. "Rio de Janeiro, Brazil",
    10. "Bali, Indonesia"
"""

## Example 1: Non-Streaming Response

Non-streaming mode returns the complete response all at once. This is simpler to work with and good for most use cases.

In [3]:
async def non_streaming_example() -> None:
    """Example of non-streaming response (get the complete result at once)."""
    print("=== Non-streaming Response Example ===\n")

    project_endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
    model_deployment = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")
    
    # For authentication, run `az login` command in terminal
    async with (
        DefaultAzureCredential() as credential,
        AzureAIProjectAgentProvider(
            credential=credential,
            project_endpoint=project_endpoint
        ) as provider,
    ):
        agent = await provider.create_agent(
            name="VacationPlannerAgent",
            instructions=agent_instructions,
            model=model_deployment,
        )

        query = "Can you help me plan a vacation to New York?"
        print(f"User: {query}")
        result = await agent.run(query)
        print(f"Agent: {result}\n")

# Run the non-streaming example
await non_streaming_example()

=== Non-streaming Response Example ===

User: Can you help me plan a vacation to New York?
Agent: Absolutely! New York City is an incredible destination with endless activities to suit any traveler. Let’s make sure your trip is thoughtfully planned to maximize your time in the Big Apple. Here’s a step-by-step vacation planning guide:

---

### **1. Trip Details**
- **How many days are you planning to stay?**
- **Travel dates:** Do you have exact dates in mind?
- **Budget:** Are you looking for luxury, mid-range, or budget recommendations?
- **Travel companions:** Are you traveling solo, with friends, as a couple, or with family (e.g., kids)?
- **Interests:** Are you drawn to sightseeing, shopping, museums, theater, nightlife, food adventures, or something else?

---

### **2. Must-See Attractions**
Here’s a list of iconic attractions in NYC that you can mix and match based on your interests:

#### **Landmarks**
- Statue of Liberty & Ellis Island (guided tours available)
- Empire State 

## Example 2: Streaming Response

Streaming mode returns results as they are generated, providing a more interactive experience. This is useful for long responses where you want to show progress.

In [ ]:
async def streaming_example() -> None:
    """Example of streaming response (get results as they are generated)."""
    print("=== Streaming Response Example ===\n")

    project_endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
    model_deployment = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")
    
    # For authentication, run `az login` command in terminal
    async with (
        DefaultAzureCredential() as credential,
        AzureAIProjectAgentProvider(
            credential=credential,
            project_endpoint=project_endpoint
        ) as provider,
    ):
        agent = await provider.create_agent(
            name="VacationPlannerAgent",
            instructions=agent_instructions,
            model=model_deployment,
        )

        query = "Can you help me plan a vacation to Paris?"
        print(f"User: {query}")
        print("Agent: ", end="", flush=True)
        async for chunk in agent.run_stream(query):
            if chunk.text:
                print(chunk.text, end="", flush=True)
        print("\n")

# Run the streaming example
await streaming_example()

## Try Your Own Queries

Experiment with different weather queries or modify the agent to do something else!

In [ ]:
# Try your own query here
async def custom_query(question: str, use_streaming: bool = False) -> None:
    """Run a custom query against the weather agent."""
    project_endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
    model_deployment = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")
    
    async with (
        DefaultAzureCredential() as credential,
        AzureAIProjectAgentProvider(
            credential=credential,
            project_endpoint=project_endpoint
        ) as provider,
    ):
        agent = await provider.create_agent(
            name="CustomWeatherAgent",
            instructions=agent_instructions,
            model=model_deployment,
        )

        print(f"User: {question}")
        
        if use_streaming:
            print("Agent: ", end="", flush=True)
            async for chunk in agent.run_stream(question):
                if chunk.text:
                    print(chunk.text, end="", flush=True)
            print()
        else:
            result = await agent.run(question)
            print(f"Agent: {result}")

# Example usage - modify the question and try it!
await custom_query("Can you help me plan a vacation to Germany? I want to learn more about its history.", use_streaming=False)

## Key Concepts

### Agent Creation
- **name**: Identifier for the agent
- **instructions**: System prompt that guides agent behavior
- **model**: The AI model deployment to use (e.g., gpt-4o)
- **tools**: Functions the agent can call

### Streaming vs Non-Streaming
- **Non-streaming** (`agent.run()`): Returns complete response at once
  - Simpler to use
  - Better for short responses
  - Easier error handling

- **Streaming** (`agent.run_stream()`): Returns response chunks as generated
  - More interactive user experience
  - Better for long responses
  - Shows progress in real-time

### Function Tools
- Defined as regular Python functions
- Use type annotations with `Annotated` and `Field` for descriptions
- Agent automatically decides when to call them
- Can pass multiple tools to an agent